In [2]:
import os
os.makedirs("src", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("docs", exist_ok=True)

In [3]:
%%writefile src/data_gen.py

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" 

import numpy as np
import torch
import os
from qiskit import QuantumCircuit
from qiskit.quantum_info import random_density_matrix, DensityMatrix
from qiskit_aer import AerSimulator

# Configuration
NUM_QUBITS = 1           # Single qubit for initial testing
NUM_SAMPLES = 1000       # Number of random states to generate
SHADOWS_PER_STATE = 500  # Number of measurement shots per state
OUTPUT_PATH = "src/shadow_dataset.pt"

def apply_pauli_rotation(qc, pauli_idx, qubit_idx):
    """
    Rotates the basis to measure in X, Y, or Z.
    0=I (Z-basis default), 1=X, 2=Y, 3=Z
    """
    if pauli_idx == 1:   # Measure in X basis
        qc.h(qubit_idx)
    elif pauli_idx == 2: # Measure in Y basis
        qc.sdg(qubit_idx)
        qc.h(qubit_idx)
    # If 3 (Z) or 0 (I), no rotation needed (standard Z-measure)
    return qc

def generate_dataset():
    simulator = AerSimulator()
    data_X = [] # Input: List of (Pauli, Outcome)
    data_Y = [] # Target: The actual density matrix rho

    print(f"Generating {NUM_SAMPLES} quantum states with {SHADOWS_PER_STATE} shadows each...")

    for i in range(NUM_SAMPLES):
        # 1. Generate a random valid density matrix (Ground Truth)
        rho_target = random_density_matrix(2**NUM_QUBITS, seed=i)
        
        # Store flattened target for training (Real + Imag parts)
        # We store it as a complex tensor
        target_tensor = torch.tensor(rho_target.data, dtype=torch.complex64)
        data_Y.append(target_tensor)

        # 2. Simulate Classical Shadows (Measurements)
        # We select random bases for this specific state
        # 1=X, 2=Y, 3=Z
        pauli_indices = np.random.randint(1, 4, size=(SHADOWS_PER_STATE, NUM_QUBITS))
        outcomes = []

        # Optimization: We can batch this, but loops are clearer for logic
        for s in range(SHADOWS_PER_STATE):
            qc = QuantumCircuit(NUM_QUBITS, NUM_QUBITS)
            
            # Initialize circuit to the random state rho
            qc.set_density_matrix(rho_target)
            
            # Apply random basis rotation
            chosen_pauli = pauli_indices[s][0] # Assuming 1 qubit
            apply_pauli_rotation(qc, chosen_pauli, 0)
            
            # Measure
            qc.measure(0, 0)
            
            # Execute
            result = simulator.run(qc, shots=1).result()
            counts = result.get_counts()
            bitstring = list(counts.keys())[0]
            
            # Convert bit '0' -> +1, bit '1' -> -1 (Eigenvalues)
            outcome = 1 if bitstring == '0' else 0 # Storing as index for embedding (0 or 1)
            outcomes.append(outcome)

        # 3. Structure the input data
        # Input shape: (SHADOWS_PER_STATE, 2) -> [Pauli_Index, Outcome_Index]
        # Pauli_Index: 1,2,3. Outcome_Index: 0,1
        input_tensor = torch.tensor(list(zip(pauli_indices.flatten(), outcomes)), dtype=torch.long)
        data_X.append(input_tensor)

        if (i+1) % 100 == 0:
            print(f"Progress: {i+1}/{NUM_SAMPLES} states generated.")

    # Convert lists to tensors
    # X shape: (NUM_SAMPLES, SHADOWS_PER_STATE, 2)
    # Y shape: (NUM_SAMPLES, 2^N, 2^N)
    dataset = {
        "x": torch.stack(data_X),
        "y": torch.stack(data_Y)
    }

    # Save to disk
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    torch.save(dataset, OUTPUT_PATH)
    print(f"Dataset saved to {OUTPUT_PATH}")

if __name__ == "__main__":
    generate_dataset()

Overwriting src/data_gen.py


In [4]:
!python src/data_gen.py

Generating 1000 quantum states with 500 shadows each...
Progress: 100/1000 states generated.
Progress: 200/1000 states generated.
Progress: 300/1000 states generated.
Progress: 400/1000 states generated.
Progress: 500/1000 states generated.
Progress: 600/1000 states generated.
Progress: 700/1000 states generated.
Progress: 800/1000 states generated.
Progress: 900/1000 states generated.
Progress: 1000/1000 states generated.
Dataset saved to src/shadow_dataset.pt


In [5]:
%%writefile src/model.py

import torch
import torch.nn as nn
import torch.nn.functional as F

class ShadowReconstructor(nn.Module):
    def __init__(self, num_qubits, embed_dim=64, num_heads=4, num_layers=2):
        super().__init__()
        self.d = 2 ** num_qubits  # Dimension of the Hilbert space
        self.output_dim = self.d ** 2  # Total parameters needed for L (complex)
        
        # 1. Embedding Layer
        # Inputs are (Pauli_Index, Outcome). 
        # Pauli indices: 0=I, 1=X, 2=Y, 3=Z. Outcome: 0=(-1), 1=(+1)
        self.pauli_embed = nn.Embedding(4, embed_dim)
        self.outcome_embed = nn.Embedding(2, embed_dim)
        
        # 2. Transformer Encoder (Process the set of shadows)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 3. Prediction Head
        self.fc_hidden = nn.Linear(embed_dim, 128)
        # We output 2 * d^2 values to account for Real and Imaginary parts of L
        self.fc_output = nn.Linear(128, self.d * self.d * 2) 

    def forward(self, paulis, outcomes):
        """
        Args:
            paulis: (batch_size, num_shadows) - Integers representing Pauli basis
            outcomes: (batch_size, num_shadows) - Integers representing measurement results
        """
        # A. Embed and Combine
        x = self.pauli_embed(paulis) + self.outcome_embed(outcomes)
        
        # B. Self-Attention (Learn correlations between snapshots)
        x = self.transformer(x)
        
        # C. Global Pooling (Enforce Permutation Invariance)
        # We average over the 'num_shadows' dimension
        x = x.mean(dim=1) 
        
        # D. Project to Matrix Space
        x = F.relu(self.fc_hidden(x))
        flat_params = self.fc_output(x)
        
        return flat_params

    def get_density_matrix(self, flat_params):
        """
        Reconstructs rho strictly using the Cholesky decomposition as per assignment Part 2.
        Formula: rho = (L @ L.H) / Tr(L @ L.H)
        """
        batch_size = flat_params.shape[0]
        
        # 1. Reshape into Real and Imaginary components
        params = flat_params.view(batch_size, self.d, self.d, 2)
        L_real = params[..., 0]
        L_imag = params[..., 1]
        
        # 2. Enforce Lower Triangular Structure
        # We zero out the upper triangle to make L strictly lower triangular
        tril_mask = torch.tril(torch.ones(self.d, self.d, device=flat_params.device))
        L_real = L_real * tril_mask
        L_imag = L_imag * tril_mask
        
        # 3. Construct Complex L
        L = torch.complex(L_real, L_imag)
        
        # 4. Compute Raw Density Matrix (rho_raw = L @ L_dagger)
        # This guarantees Hermiticity and PSD
        L_dagger = L.mH # Conjugate transpose
        rho_raw = torch.matmul(L, L_dagger)
        
        # 5. Enforce Unit Trace
        trace = torch.diagonal(rho_raw, dim1=-2, dim2=-1).sum(-1)
        rho = rho_raw / trace.view(-1, 1, 1)
        
        return rho

Overwriting src/model.py


In [12]:
%%writefile src/train.py

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import time
import numpy as np
from model import ShadowReconstructor 

# --- Configuration ---
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
EPOCHS = 10
DATA_PATH = "src/shadow_dataset.pt"
MODEL_SAVE_PATH = "outputs/model_weights.pt"

# --- 1. Dataset Wrapper ---
class QuantumShadowDataset(Dataset):
    def __init__(self, filepath):
        data = torch.load(filepath)
        self.x = data['x']
        self.y = data['y']
        
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

# --- 2. Metric Helpers ---
def compute_metrics(rho_pred, rho_true):
    """
    Computes both Fidelity and Trace Distance.
    """
    # Move to CPU and high precision complex128
    rho = rho_pred.detach().cpu().to(torch.complex128)
    sigma = rho_true.detach().cpu().to(torch.complex128)
    
    batch_fid = []
    batch_trace_dist = []
    
    for i in range(rho.shape[0]):
        r = rho[i]
        s = sigma[i]
        
        # --- Metric 1: Quantum Fidelity ---
        # 1. Calculate sqrt(rho)
        evals_r, evecs_r = torch.linalg.eigh(r)
        evals_r = torch.clamp(evals_r, min=0)
        # Fix: Force diagonal to be complex
        diag_r = torch.diag(torch.sqrt(evals_r)).to(torch.complex128)
        sqrt_r = evecs_r @ diag_r @ evecs_r.mH
        
        # 2. Calculate product: sqrt(rho) * sigma * sqrt(rho)
        temp = sqrt_r @ s @ sqrt_r
        
        # 3. Calculate sqrt of that product
        evals_t, evecs_t = torch.linalg.eigh(temp)
        evals_t = torch.clamp(evals_t, min=0)
        diag_t = torch.diag(torch.sqrt(evals_t)).to(torch.complex128)
        sqrt_temp = evecs_t @ diag_t @ evecs_t.mH
        
        # Trace and square
        trace_val = torch.real(torch.trace(sqrt_temp))
        fidelity = trace_val ** 2
        batch_fid.append(fidelity.item())
        
        # --- Metric 2: Trace Distance ---
        # T(rho, sigma) = 0.5 * sum(|eigenvalues(rho - sigma)|)
        # Since rho and sigma are Hermitian, their difference is Hermitian.
        # We can use eigvalsh (stable for Hermitian matrices).
        diff = r - s
        evals_diff = torch.linalg.eigvalsh(diff)
        trace_dist = 0.5 * torch.sum(torch.abs(evals_diff))
        batch_trace_dist.append(trace_dist.item())
        
    return np.mean(batch_fid), np.mean(batch_trace_dist)

# --- 3. Training Loop ---
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on {device}")
    
    dataset = QuantumShadowDataset(DATA_PATH)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_data, test_data = torch.utils.data.random_split(dataset, [train_size, test_size])
    
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)
    
    model = ShadowReconstructor(num_qubits=1).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss() 
    
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            paulis = batch_x[:, :, 0]
            outcomes = batch_x[:, :, 1]
            
            optimizer.zero_grad()
            flat_params = model(paulis, outcomes)
            rho_pred = model.get_density_matrix(flat_params)
            
            loss = criterion(rho_pred.real, batch_y.real) + criterion(rho_pred.imag, batch_y.imag)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        # Evaluation
        model.eval()
        total_fidelity = 0
        total_trace_dist = 0
        start_time = time.time()
        
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x = batch_x.to(device)
                paulis = batch_x[:, :, 0]
                outcomes = batch_x[:, :, 1]
                
                flat_params = model(paulis, outcomes)
                rho_pred = model.get_density_matrix(flat_params)
                
                fid, td = compute_metrics(rho_pred, batch_y)
                total_fidelity += fid
                total_trace_dist += td
        
        avg_fidelity = total_fidelity / len(test_loader)
        avg_trace_dist = total_trace_dist / len(test_loader)
        
        # Calculate Inference Latency per sample
        # Total time / (number of batches * batch_size)
        total_samples = len(test_loader) * BATCH_SIZE
        latency_ms = ((time.time() - start_time) / total_samples) * 1000
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Fidelity: {avg_fidelity:.4f} | TraceDist: {avg_trace_dist:.4f} | Latency: {latency_ms:.2f}ms")

    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"Model saved to {MODEL_SAVE_PATH}")

if __name__ == "__main__":
    train()

Overwriting src/train.py


In [16]:
import sys
import os

# 1. Fix the library conflict (Just to be safe)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# 2. Add 'src' to the python path so we can import files from it
sys.path.append(os.path.abspath("src"))

# 3. Import the train function from your file
from src.train import train

# 4. Run it!
train()

Training on cpu
Epoch 1/10 | Loss: 0.0817 | Fidelity: 0.8017 | TraceDist: 0.3642 | Latency: 12.97ms
Epoch 2/10 | Loss: 0.0551 | Fidelity: 0.8880 | TraceDist: 0.2689 | Latency: 6.93ms
Epoch 3/10 | Loss: 0.0250 | Fidelity: 0.9665 | TraceDist: 0.1331 | Latency: 4.85ms
Epoch 4/10 | Loss: 0.0063 | Fidelity: 0.9825 | TraceDist: 0.0868 | Latency: 4.96ms
Epoch 5/10 | Loss: 0.0046 | Fidelity: 0.9839 | TraceDist: 0.0840 | Latency: 5.84ms
Epoch 6/10 | Loss: 0.0039 | Fidelity: 0.9827 | TraceDist: 0.0876 | Latency: 5.29ms
Epoch 7/10 | Loss: 0.0033 | Fidelity: 0.9860 | TraceDist: 0.0736 | Latency: 4.72ms
Epoch 8/10 | Loss: 0.0033 | Fidelity: 0.9875 | TraceDist: 0.0726 | Latency: 7.34ms
Epoch 9/10 | Loss: 0.0030 | Fidelity: 0.9861 | TraceDist: 0.0802 | Latency: 5.44ms
Epoch 10/10 | Loss: 0.0030 | Fidelity: 0.9872 | TraceDist: 0.0770 | Latency: 5.63ms
Model saved to outputs/model_weights.pt


In [ ]:
%%writefile README.md
# QCG PaAC Open Project: Density Matrix Reconstruction

**Track 1: Classical Shadows with Transformer Architecture**

## Project Overview
This project implements a machine learning model capable of reconstructing a quantum density matrix $\rho$ from measurement data (Classical Shadows). The model enforces strict physical constraints (Hermitian, Positive Semi-Definite, Unit Trace) using a Cholesky decomposition approach.

## Repository Structure
* **/src**: Contains core source code for the model (`model.py`), data generation (`data_gen.py`), and training loops (`train.py`).
* **/outputs**: Stores saved model weights (`model_weights.pt`) and training logs.
* **/docs**: Detailed technical documentation and replication guides.

## Performance Metrics
Epoch 1/10 | Loss: 0.0817 | Fidelity: 0.8017 | TraceDist: 0.3642 | Latency: 12.97ms
Epoch 2/10 | Loss: 0.0551 | Fidelity: 0.8880 | TraceDist: 0.2689 | Latency: 6.93ms
Epoch 3/10 | Loss: 0.0250 | Fidelity: 0.9665 | TraceDist: 0.1331 | Latency: 4.85ms
Epoch 4/10 | Loss: 0.0063 | Fidelity: 0.9825 | TraceDist: 0.0868 | Latency: 4.96ms
Epoch 5/10 | Loss: 0.0046 | Fidelity: 0.9839 | TraceDist: 0.0840 | Latency: 5.84ms
Epoch 6/10 | Loss: 0.0039 | Fidelity: 0.9827 | TraceDist: 0.0876 | Latency: 5.29ms
Epoch 7/10 | Loss: 0.0033 | Fidelity: 0.9860 | TraceDist: 0.0736 | Latency: 4.72ms
Epoch 8/10 | Loss: 0.0033 | Fidelity: 0.9875 | TraceDist: 0.0726 | Latency: 7.34ms
Epoch 9/10 | Loss: 0.0030 | Fidelity: 0.9861 | TraceDist: 0.0802 | Latency: 5.44ms
Epoch 10/10 | Loss: 0.0030 | Fidelity: 0.9872 | TraceDist: 0.0770 | Latency: 5.63ms
Final trained model | Loss: 0.0030 | Fidelity: 0.9872 | TraceDist: 0.0770 | Latency: 5.63ms 
Model saved to outputs/model_weights.pt. View metrics there

## AI Attribution Policy
In compliance with the QCG PaAC Open Project guidelines:
* **Tools Used**: Google Gemini (https://gemini.google.com/share/82a098e65b9e)
* **Usage**:
    * Generated PyTorch boilerplate for the `ShadowReconstructor` class.
    * Debugged tensor shape mismatches in Cholesky decomposition.
* **Verification**:
    * Math verified against standard quantum mechanics textbooks ($\rho = LL^{\dagger}$).
    * Fidelity metric cross-referenced with standard library implementations.
    * Final results say it all. A Fidelity of 98.72% is considered really good in QST. A trace distance value of 0.077 is very low, corroborating the high fidelity score. The loss drops rapidly from 0.0817 to 0.0030 and stabilizes suggesting that the model has finished learning. A latency of 5-6 milliseconds is very fast, showing that the model can quickly recognize the states now.

Overwriting README.md
